# Download Duck land elevation data and combine into one .nc-file

## Download data
Code produced mainly by Duck AI chat with GPT-4o

In [1]:
import requests
import xml.etree.ElementTree as ET
import os

import xarray as xr

In [2]:
path = '/home/jovyan/data/98_paper2_kf_dtm/02_Duck_validation_data/'

In [3]:
# Find existing files
existing_files = []
for root, dirs, files in os.walk(path):
	for fname in files:
		if fname.endswith('.nc'):
			existing_files.append(fname)

In [4]:
# Find all files in xml catalog
all_dataset_names = []

# URL of the THREDDS XML catalog
thredds_catalog_url = 'https://chldata.erdc.dren.mil/thredds/catalog/frf/geomorphology/DEMs/surveyDEM/data/catalog.xml'

# Send a GET request to fetch the XML catalog
response = requests.get(thredds_catalog_url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the XML content
    root = ET.fromstring(response.content)
    # namespace
    ns = {'thredds': 'http://www.unidata.ucar.edu/namespaces/thredds/InvCatalog/v1.0'}

    # Find all datasets
    for dataset in root.findall('.//thredds:dataset', ns):
        all_dataset_names.append(dataset.get('name'))
else:
    print("Failed to fetch XML catalog:", response.status_code)

In [6]:
# Download the files from the server, that are not existing yet in the folder
thredds_server = 'https://chldata.erdc.dren.mil/thredds/fileServer/frf/geomorphology/DEMs/surveyDEM/data/'
for dataset_name in all_dataset_names[1:]:
    if dataset_name in existing_files:
        continue
    else:
        full_dataset_url = f"{thredds_server}/{dataset_name}"
        print(f"Downloading {dataset_name} from {full_dataset_url}...")
        file_response = requests.get(full_dataset_url)
        if file_response.status_code == 200:
            # Define the local file path
            local_file_path = os.path.join(path, dataset_name)

            # Save the file
            with open(local_file_path, 'wb') as f:
                f.write(file_response.content)
            print(f"Downloaded {dataset_name} successfully to {local_file_path}.")
        else:
            print(f"Failed to download {full_dataset_url}: {file_response.status_code}")

Downloaded FRF_geomorphology_DEMs_surveyDEM_20250310.nc successfully to /home/jovyan/data/98_paper2_kf_dtm/02_Duck_validation_data/FRF_geomorphology_DEMs_surveyDEM_20250310.nc.
Downloaded FRF_geomorphology_DEMs_surveyDEM_20250415.nc successfully to /home/jovyan/data/98_paper2_kf_dtm/02_Duck_validation_data/FRF_geomorphology_DEMs_surveyDEM_20250415.nc.


## Merge all .nc-files

In [7]:
fn = 'duck_validation.nc'
if (not os.path.isfile(path+fn)):
    all_data = xr.Dataset()

    for root, dirs, files in os.walk(path):        
        for fname in files:
            if fname.endswith('.nc'):
                # print(fname)
                try:
                    data_temp = xr.open_dataset(os.path.join(root, fname))
                except:
                    print(f'Failed loading {fname}')
                try:
                    all_data = all_data.merge(data_temp, compat='override')
                except:
                    print(f'Failed to merge {fname}')

    all_data.to_netcdf(path+fn)
else:
    all_data = xr.open_dataset(path+fn)

From end of 2019 on, longitude and latitudes are about 1e-6 different from before (x and y are the same), which translates to a few decimeters difference. Therefore the compat='override' parameter. 
  
There were daily observations in 09/1997 so will not try to figure out why this file is not loading.